<a href="https://colab.research.google.com/github/mohamadfaisalbashir/Practical-Linear-Algebra/blob/main/06_Matrices_Part_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Matrices, Part 2**

This notebook covers Matrices, Part 2:
1. Matrix Norms
2. Matrix Spaces (Column, Row, Nulls)
3. Rank
4. Determinant
5. Summary

## **1. Matrix Norms**

You learned about vector norms in Chapter 2: the norm of a vector is its Euclidean geometric length, which is computed as the square root of the sum of the squared vector elements. Matrix norms are a little more complicated. For one thing, there is no "the" matrix norm; there are multiple distinct norms that can be computed from a matrix. Matrix norms are somewhat similar to vector norms in that each norm provides one number that characterizes a matrix, and the norm is indicated using double-vertical lines, as in the norm of matrix A indicated as ||A||.

However, different matrix norms have different meanings. The myriad of matrix norms can be broadly divided into two families: element-wise (also called entry-wise) and induced. Element-wise norms are computed based on the individual elements of the matrix, and thus these norms can be interpreted to reflect the magnitudes of the elements in the matrix. Induced norms, on the other hand, can be interpreted in the following way: one of the functions of a matrix is to encode a transformation of a vector; the induced norm of a matrix is a measure of how much that transformation scales (stretches or shrinks) that vector. This interpretation will make more sense in Chapter 7 when you learn about using matrices for geometric transformations, and in Chapter 14 when you learn about the singular value decomposition.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg

# Set random seed for reproducibility
np.random.seed(42)

### **1.1 Matrix Trace and Frobenius Norm**

The Euclidean norm, also called the Frobenius norm, is computed as the square root of the sum of all matrix elements squared. This is a direct extension of the vector norm to matrices. The Frobenius norm is indicated with a subscripted F as ||A||_F and is calculated according to the equation:

||A||_F = √(∑ᵢ₌₁ᴹ ∑ⱼ₌₁ᴺ aᵢⱼ²)

where the indices i and j correspond to the M rows and N columns. The Frobenius norm is also called the ℓ2 norm (the ℓ is a fancy-looking letter L). The ℓ2 norm gets its name from the general formula for element-wise p-norms:

||A||_p = (∑ᵢ₌₁ᴹ ∑ⱼ₌₁ᴺ |aᵢⱼ|^p)^(1/p)

Notice that you get the Frobenius norm when p = 2. The matrix trace, denoted as tr(A), is the sum of the diagonal elements of a square matrix. There is a beautiful relationship between the trace and the Frobenius norm: the square of the Frobenius norm equals the trace of A^T A, that is: ||A||_F² = tr(A^T A). Matrix norms have several applications in machine learning and statistical analysis. One important application is in regularization, which aims to improve model fitting and increase generalization of models to unseen data. The basic idea of regularization is to add a matrix norm as a cost function to a minimization algorithm. That norm will help prevent model parameters from becoming too large (ℓ2 regularization, also called ridge regression) or encourage sparse solutions (ℓ1 regularization, also called lasso regression). In modern deep learning, matrix norms are essential for achieving impressive performance at solving computer vision problems.

Another application of the Frobenius norm is computing a measure of "matrix distance." The distance between a matrix and itself is 0, and the distance between two distinct matrices increases as the numerical values in those matrices become increasingly dissimilar. Frobenius matrix distance is computed simply by replacing the matrix with the difference of two matrices.

In [2]:
# Create a sample matrix
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

print("Matrix A:")
print(A)

# Calculate Frobenius norm using numpy
frobenius_norm = np.linalg.norm(A, 'fro')
print(f"\nFrobenius norm using numpy: {frobenius_norm:.4f}")

# Calculate Frobenius norm manually
frobenius_manual = np.sqrt(np.sum(A**2))
print(f"Frobenius norm calculated manually: {frobenius_manual:.4f}")

# Calculate using trace method: ||A||_F² = tr(A^T A)
trace_method = np.sqrt(np.trace(A.T @ A))
print(f"Frobenius norm using trace method: {trace_method:.4f}")

# Verify they are all equal
print(f"\nAll methods agree: {np.allclose(frobenius_norm, frobenius_manual) and np.allclose(frobenius_norm, trace_method)}")

Matrix A:
[[1 2 3]
 [4 5 6]
 [7 8 9]]

Frobenius norm using numpy: 16.8819
Frobenius norm calculated manually: 16.8819
Frobenius norm using trace method: 16.8819

All methods agree: True


In [3]:
# Calculate the trace
trace_A = np.trace(A)
print(f"Trace of A (sum of diagonal elements): {trace_A}")
print(f"Diagonal elements: {np.diag(A)}")
print(f"Manual sum of diagonal: {np.sum(np.diag(A))}")

# Verify the relationship: ||A||_F² = tr(A^T A)
frobenius_squared = frobenius_norm ** 2
trace_ATA = np.trace(A.T @ A)
print(f"\n||A||_F² = {frobenius_squared:.4f}")
print(f"tr(A^T A) = {trace_ATA:.4f}")
print(f"These are equal: {np.isclose(frobenius_squared, trace_ATA)}")

Trace of A (sum of diagonal elements): 15
Diagonal elements: [1 5 9]
Manual sum of diagonal: 15

||A||_F² = 285.0000
tr(A^T A) = 285.0000
These are equal: True


In [4]:
# Calculate different p-norms
print("Different p-norms of matrix A:")
print(f"p=1 (nuclear norm): {np.linalg.norm(A, 1):.4f}")
print(f"p=2 (Frobenius norm): {np.linalg.norm(A, 2):.4f}")
print(f"p=∞ (spectral norm): {np.linalg.norm(A, np.inf):.4f}")

# Manual calculation of p=1 norm (max absolute column sum)
p1_norm = np.max(np.sum(np.abs(A), axis=0))
print(f"\np=1 norm (max column sum) verified: {p1_norm:.4f}")

Different p-norms of matrix A:
p=1 (nuclear norm): 18.0000
p=2 (Frobenius norm): 16.8481
p=∞ (spectral norm): 24.0000

p=1 norm (max column sum) verified: 18.0000


In [5]:
# Matrix distance using Frobenius norm
B = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 10]])  # Slightly different from A

# Distance is the Frobenius norm of the difference
distance = np.linalg.norm(A - B, 'fro')
print(f"Matrix A:")
print(A)
print(f"\nMatrix B:")
print(B)
print(f"\nFrobenius distance between A and B: {distance:.4f}")

# The difference matrix
print(f"\nDifference A - B:")
print(A - B)

# Distance from a matrix to itself should be 0
distance_self = np.linalg.norm(A - A, 'fro')
print(f"\nDistance from A to itself: {distance_self:.4f}")

Matrix A:
[[1 2 3]
 [4 5 6]
 [7 8 9]]

Matrix B:
[[ 1  2  3]
 [ 4  5  6]
 [ 7  8 10]]

Frobenius distance between A and B: 1.0000

Difference A - B:
[[ 0  0  0]
 [ 0  0  0]
 [ 0  0 -1]]

Distance from A to itself: 0.0000


## **2. Matrix Spaces (Column, Row, Nulls)**

Matrix spaces are an extension of the vector subspaces concept from Chapter 3. A matrix space is the set of all vectors that can be created as linear combinations of the columns (or rows) of a matrix, or the set of vectors that satisfy particular matrix equations. There are four fundamental matrix spaces, and they all arise naturally from a single matrix. Understanding these spaces is crucial for understanding rank, which in turn is crucial for understanding inverse, least squares, and eigendecomposition.

The four fundamental matrix spaces are:
1. **Column Space** (also called the range): The set of all possible linear combinations of the matrix columns.
2. **Row Space**: The set of all possible linear combinations of the matrix rows.
3. **Null Space**: The set of all vectors that, when multiplied by the matrix, produce the zero vector.
4. **Left Null Space**: The set of all vectors that, when multiplied (from the left) by the transpose of the matrix, produce the zero vector.

These spaces have dimensions and orthogonal relationships that define the structure of the matrix. Understanding these spaces allows us to understand why certain matrix equations have solutions, and how to find those solutions.

### **2.1 Column Space**

The column space of a matrix A, denoted C(A) or the range of A, is the set of all vectors that can be expressed as a linear combination of the columns of A. In mathematical terms, the column space is the span of the columns of the matrix. The column space is also called the image or the range of the matrix.

To understand the column space, consider that multiplying a matrix by a vector is equivalent to taking a linear combination of the columns of the matrix. That is, if we have a matrix A with columns a₁, a₂, ..., aₙ, and a vector x with elements x₁, x₂, ..., xₙ, then:

Ax = x₁a₁ + x₂a₂ + ... + xₙaₙ

Therefore, the column space consists of all vectors b such that the equation Ax = b has at least one solution. The dimension of the column space is the rank of the matrix, which we will discuss in detail in the section on rank.

In [6]:
# Example of column space
A = np.array([[1, 2],
              [2, 4],
              [3, 6]])

print("Matrix A:")
print(A)
print(f"\nColumns of A:")
print(f"Column 1: {A[:, 0]}")
print(f"Column 2: {A[:, 1]}")

# Notice that column 2 is 2 times column 1
print(f"\nColumn 2 = 2 * Column 1: {np.allclose(A[:, 1], 2 * A[:, 0])}")

# The column space is 1-dimensional (spanned by column 1)
# Let's verify by checking if some vectors are in the column space
# A vector b is in C(A) if Ax = b has a solution

# Try some vectors
test_vectors = [
    np.array([1, 2, 3]),  # This is column 1
    np.array([2, 4, 6]),  # This is column 2 (= 2 * column 1)
    np.array([5, 10, 15]), # This is 5 * column 1
    np.array([1, 1, 1])   # This is not in column space
]

for i, b in enumerate(test_vectors):
    try:
        x = np.linalg.lstsq(A, b, rcond=None)[0]
        residual = np.linalg.norm(A @ x - b)
        print(f"\nVector {i}: {b}")
        print(f"Residual: {residual:.6f}")
        if residual < 1e-10:
            print(f"Vector {i} is in the column space (solution: x = {x})")
        else:
            print(f"Vector {i} is NOT in the column space")
    except:
        print(f"\nVector {i}: {b} - Could not find exact solution")

Matrix A:
[[1 2]
 [2 4]
 [3 6]]

Columns of A:
Column 1: [1 2 3]
Column 2: [2 4 6]

Column 2 = 2 * Column 1: True

Vector 0: [1 2 3]
Residual: 0.000000
Vector 0 is in the column space (solution: x = [0.2 0.4])

Vector 1: [2 4 6]
Residual: 0.000000
Vector 1 is in the column space (solution: x = [0.4 0.8])

Vector 2: [ 5 10 15]
Residual: 0.000000
Vector 2 is in the column space (solution: x = [1. 2.])

Vector 3: [1 1 1]
Residual: 0.654654
Vector 3 is NOT in the column space


### **2.2 Row Space**

The row space of a matrix A, denoted R(A), is the set of all vectors that can be expressed as a linear combination of the rows of A. The row space is the span of the rows of the matrix. Importantly, the row space of matrix A is equivalent to the column space of matrix A^T.

In mathematical terms:
R(A) = C(A^T)

This relationship means that we can analyze the row space by working with the column space of the transpose. The dimension of the row space is also equal to the rank of the matrix. In fact, the rank of the row space equals the rank of the column space—this is one of the fundamental theorems of linear algebra that makes the concept of rank so powerful.

The row space is particularly important in applications where we interpret the rows of a matrix as data samples or observations, such as in statistical analysis and machine learning.

In [7]:
# Example of row space
A = np.array([[1, 2, 3],
              [2, 4, 6],
              [3, 5, 7]])

print("Matrix A:")
print(A)
print(f"\nTranspose A^T:")
print(A.T)

print(f"\nRows of A:")
for i in range(A.shape[0]):
    print(f"Row {i}: {A[i]}")

# Note: Row 2 = 2 * Row 1
print(f"\nRow 2 = 2 * Row 1: {np.allclose(A[1], 2 * A[0])}")

# The row space of A is the column space of A^T
print(f"\nRow space of A = Column space of A^T")
print(f"This means a vector v is in R(A) if A^T @ x = v has a solution")

# Verify with an example
# A linear combination of rows: 1 * Row 0 + 2 * Row 1 + 3 * Row 2
coefficients = np.array([1, 2, 3])
linear_combo = coefficients @ A  # This gives a vector in the row space

print(f"\nLinear combination of rows: 1*Row0 + 2*Row1 + 3*Row2 = {linear_combo}")
print(f"This vector is in R(A)")

# We can find the coefficients by solving A^T @ c = v
v = linear_combo
c_recovered = np.linalg.lstsq(A.T, v, rcond=None)[0]
print(f"\nVerification: solving A^T @ c = v gives c = {c_recovered}")
print(f"These match our original coefficients: {np.allclose(c_recovered, coefficients)}")

Matrix A:
[[1 2 3]
 [2 4 6]
 [3 5 7]]

Transpose A^T:
[[1 2 3]
 [2 4 5]
 [3 6 7]]

Rows of A:
Row 0: [1 2 3]
Row 1: [2 4 6]
Row 2: [3 5 7]

Row 2 = 2 * Row 1: True

Row space of A = Column space of A^T
This means a vector v is in R(A) if A^T @ x = v has a solution

Linear combination of rows: 1*Row0 + 2*Row1 + 3*Row2 = [14 25 36]
This vector is in R(A)

Verification: solving A^T @ c = v gives c = [1. 2. 3.]
These match our original coefficients: True


### **2.3 Null Spaces**

The null space of a matrix A, denoted N(A), is the set of all vectors x such that Ax = 0. In other words, the null space consists of all solutions to the homogeneous system of linear equations Ax = 0. The null space is also called the kernel of the matrix.

The null space is important because it tells us about the dependencies in the matrix. For example:
- If the null space contains only the zero vector, then the columns of A are linearly independent.
- If the null space contains non-zero vectors, then the columns of A are linearly dependent.
- The dimension of the null space is called the nullity of the matrix, which equals n - rank(A), where n is the number of columns.

The left null space of a matrix A, denoted N(A^T), is the set of all vectors y such that y^T A = 0 (or equivalently, A^T y = 0). The left null space is the null space of the transpose, and its dimension equals m - rank(A), where m is the number of rows.

These four fundamental spaces have a profound relationship: the row space and the left null space are orthogonal complements in ℝᵐ, and the column space and the null space are orthogonal complements in ℝⁿ. This is the fundamental theorem of linear algebra.

In [8]:
# Example of null space
A = np.array([[1, 2, 3],
              [2, 4, 6],
              [1, 1, 1]])

print("Matrix A:")
print(A)

# Find the null space using SVD
U, S, Vt = np.linalg.svd(A)

print(f"\nSingular values: {S}")
print(f"Rank of A: {np.sum(S > 1e-10)}")

# Null space vectors correspond to zero singular values
null_space_threshold = 1e-10
null_space_vectors = Vt[np.sum(S > null_space_threshold):, :].T

print(f"\nNull space dimension (nullity): {null_space_vectors.shape[1]}")
if null_space_vectors.shape[1] > 0:
    print(f"Null space basis vectors:")
    for i in range(null_space_vectors.shape[1]):
        v = null_space_vectors[:, i]
        print(f"\nNull space vector {i}: {v}")
        print(f"A @ v = {A @ v}")
        print(f"||A @ v||: {np.linalg.norm(A @ v):.10f}")
else:
    print("The null space contains only the zero vector (columns are linearly independent)")

Matrix A:
[[1 2 3]
 [2 4 6]
 [1 1 1]]

Singular values: [8.51978293e+00 6.42883231e-01 7.52824334e-16]
Rank of A: 2

Null space dimension (nullity): 1
Null space basis vectors:

Null space vector 0: [ 0.40824829 -0.81649658  0.40824829]
A @ v = [1.33226763e-15 2.66453526e-15 1.11022302e-15]
||A @ v||: 0.0000000000


In [9]:
# Left null space: N(A^T) = set of vectors y such that A^T @ y = 0
print("Left null space (null space of A^T):")
print("We want vectors y such that A^T @ y = 0")

# Find null space of A^T
U, S, Vt = np.linalg.svd(A.T)

print(f"\nA^T shape: {A.T.shape}")
print(f"Singular values of A^T: {S}")
print(f"Rank of A^T: {np.sum(S > 1e-10)}")

# Left null space vectors correspond to zero singular values of A^T
left_null_vectors = Vt[np.sum(S > null_space_threshold):, :].T

print(f"\nLeft null space dimension: {left_null_vectors.shape[1]}")

if left_null_vectors.shape[1] > 0:
    print(f"Left null space basis vectors:")
    for i in range(left_null_vectors.shape[1]):
        y = left_null_vectors[:, i]
        print(f"\nLeft null space vector {i}: {y}")
        print(f"A^T @ y = {A.T @ y}")
        print(f"||A^T @ y||: {np.linalg.norm(A.T @ y):.10f}")
else:
    print("The left null space contains only the zero vector")

# Verify the rank-nullity theorem: rank + nullity = n (number of columns)
rank_A = np.sum(np.linalg.svd(A)[1] > 1e-10)
nullity_A = null_space_vectors.shape[1]
n_columns = A.shape[1]

print(f"\n=== Rank-Nullity Theorem ===")
print(f"Rank of A: {rank_A}")
print(f"Nullity of A: {nullity_A}")
print(f"Number of columns: {n_columns}")
print(f"Rank + Nullity = {rank_A} + {nullity_A} = {rank_A + nullity_A} = {n_columns}: {rank_A + nullity_A == n_columns}")

Left null space (null space of A^T):
We want vectors y such that A^T @ y = 0

A^T shape: (3, 3)
Singular values of A^T: [8.51978293e+00 6.42883231e-01 2.60081559e-16]
Rank of A^T: 2

Left null space dimension: 1
Left null space basis vectors:

Left null space vector 0: [-8.94427191e-01  4.47213595e-01 -1.26287869e-15]
A^T @ y = [-8.18789481e-16 -3.74700271e-16  6.93889390e-17]
||A^T @ y||: 0.0000000000

=== Rank-Nullity Theorem ===
Rank of A: 2
Nullity of A: 1
Number of columns: 3
Rank + Nullity = 2 + 1 = 3 = 3: True


## **3. Rank**

The rank of a matrix is one of the most important concepts in linear algebra. The rank of a matrix A, denoted rank(A), is the dimension of the column space of A, which is also equal to the dimension of the row space of A. In simpler terms, the rank is the number of linearly independent rows or columns in the matrix.

The rank provides critical information about the matrix:
- A matrix has full rank if rank(A) = min(m, n), where m is the number of rows and n is the number of columns.
- If rank(A) < min(m, n), the matrix is rank-deficient.
- A square matrix is invertible if and only if it has full rank (rank = n for an n×n matrix).
- The rank determines the dimension of the null space through the rank-nullity theorem: rank(A) + nullity(A) = n.

Understanding rank is essential for solving systems of linear equations, finding matrix inverses, computing least squares solutions, and many other applications in linear algebra and data science.

In [10]:
# Example 1: Full rank matrix
A_full = np.array([[1, 0, 0],
                   [0, 1, 0],
                   [0, 0, 1]])

print("Example 1: Identity matrix (full rank)")
print("Matrix A:")
print(A_full)
print(f"Shape: {A_full.shape}")
print(f"Rank: {np.linalg.matrix_rank(A_full)}")
print(f"Full rank: {np.linalg.matrix_rank(A_full) == min(A_full.shape)}")

# Example 2: Rank-deficient matrix (linearly dependent columns)
A_rank_deficient = np.array([[1, 2, 3],
                            [2, 4, 6],
                            [3, 6, 9]])

print("\nExample 2: Rank-deficient matrix")
print("Matrix A (all rows are multiples of the first row):")
print(A_rank_deficient)
print(f"Shape: {A_rank_deficient.shape}")
print(f"Rank: {np.linalg.matrix_rank(A_rank_deficient)}")
print(f"Full rank: {np.linalg.matrix_rank(A_rank_deficient) == min(A_rank_deficient.shape)}")

# Verify linear dependence
print(f"\nVerifying linear dependence:")
print(f"Row 2 = 2 * Row 1: {np.allclose(A_rank_deficient[1], 2 * A_rank_deficient[0])}")
print(f"Row 3 = 3 * Row 1: {np.allclose(A_rank_deficient[2], 3 * A_rank_deficient[0])}")

# Example 3: Rectangular matrix
A_rect = np.array([[1, 2, 3, 4],
                  [5, 6, 7, 8],
                  [9, 10, 11, 12]])

print("\nExample 3: Rectangular matrix (3x4)")
print("Matrix A:")
print(A_rect)
print(f"Shape: {A_rect.shape}")
print(f"Rank: {np.linalg.matrix_rank(A_rect)}")
print(f"min(m,n): {min(A_rect.shape)}")
print(f"Full rank: {np.linalg.matrix_rank(A_rect) == min(A_rect.shape)}")

Example 1: Identity matrix (full rank)
Matrix A:
[[1 0 0]
 [0 1 0]
 [0 0 1]]
Shape: (3, 3)
Rank: 3
Full rank: True

Example 2: Rank-deficient matrix
Matrix A (all rows are multiples of the first row):
[[1 2 3]
 [2 4 6]
 [3 6 9]]
Shape: (3, 3)
Rank: 1
Full rank: False

Verifying linear dependence:
Row 2 = 2 * Row 1: True
Row 3 = 3 * Row 1: True

Example 3: Rectangular matrix (3x4)
Matrix A:
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
Shape: (3, 4)
Rank: 2
min(m,n): 3
Full rank: False


### **3.1 Ranks of Special Matrices**

Different types of matrices have specific rank properties that are useful to remember:

**Identity Matrix**: An m×m identity matrix always has rank m (full rank).

**Zero Matrix**: The zero matrix has rank 0 (no linearly independent rows or columns).

**Diagonal Matrix**: A diagonal matrix has rank equal to the number of non-zero diagonal elements.

**Rank-1 Matrix**: A rank-1 matrix can be expressed as the outer product of two non-zero vectors: A = uv^T. All rows are multiples of each other, and all columns are multiples of each other.

**Transpose**: The rank of a matrix equals the rank of its transpose: rank(A) = rank(A^T).

**Matrix Products**: rank(AB) ≤ min(rank(A), rank(B)). The rank of a product is at most the minimum rank of the factors.

In [11]:
print("Rank of Special Matrices\n")
print("=" * 50)

# 1. Identity Matrix
print("\n1. Identity Matrix")
I = np.eye(4)
print(f"4x4 Identity matrix has rank: {np.linalg.matrix_rank(I)}")
print(f"Expected: 4 (full rank)")

# 2. Zero Matrix
print("\n2. Zero Matrix")
Z = np.zeros((3, 4))
print(f"3x4 Zero matrix has rank: {np.linalg.matrix_rank(Z)}")
print(f"Expected: 0")

# 3. Diagonal Matrix
print("\n3. Diagonal Matrix")
D = np.diag([1, 2, 0, 3])
print(f"Diagonal matrix:\n{D}")
print(f"Diagonal elements: {np.diag(D)}")
print(f"Rank: {np.linalg.matrix_rank(D)}")
print(f"Expected: 3 (number of non-zero diagonal elements)")

# 4. Rank-1 Matrix
print("\n4. Rank-1 Matrix")
u = np.array([1, 2, 3]).reshape(-1, 1)
v = np.array([2, 3, 4]).reshape(-1, 1)
rank_1 = u @ v.T
print(f"u = \n{u}")
print(f"v = \n{v}")
print(f"A = u @ v^T = \n{rank_1}")
print(f"Rank of A: {np.linalg.matrix_rank(rank_1)}")
print(f"Expected: 1")

# Verify all rows are multiples
print(f"Row ratios:")
for i in range(1, rank_1.shape[0]):
    ratio = rank_1[i] / rank_1[0]
    print(f"Row {i} / Row 0 = {ratio}")

# 5. Rank of Transpose
print("\n5. Rank of Transpose")
A = np.array([[1, 2, 3],
              [4, 5, 6]])
print(f"Matrix A (2x3):\n{A}")
print(f"rank(A) = {np.linalg.matrix_rank(A)}")
print(f"rank(A^T) = {np.linalg.matrix_rank(A.T)}")
print(f"They are equal: {np.linalg.matrix_rank(A) == np.linalg.matrix_rank(A.T)}")

# 6. Rank of Matrix Products
print("\n6. Rank of Matrix Products")
A = np.array([[1, 2],
              [3, 4],
              [5, 6]])
B = np.array([[1, 0, 1],
              [0, 1, 1]])
print(f"Matrix A (3x2), rank = {np.linalg.matrix_rank(A)}")
print(f"Matrix B (2x3), rank = {np.linalg.matrix_rank(B)}")
AB = A @ B
print(f"A @ B (3x3), rank = {np.linalg.matrix_rank(AB)}")
print(f"rank(AB) ≤ min(rank(A), rank(B)) = min({np.linalg.matrix_rank(A)}, {np.linalg.matrix_rank(B)}) = {min(np.linalg.matrix_rank(A), np.linalg.matrix_rank(B))}")
print(f"Verified: {np.linalg.matrix_rank(AB) <= min(np.linalg.matrix_rank(A), np.linalg.matrix_rank(B))}")

Rank of Special Matrices


1. Identity Matrix
4x4 Identity matrix has rank: 4
Expected: 4 (full rank)

2. Zero Matrix
3x4 Zero matrix has rank: 0
Expected: 0

3. Diagonal Matrix
Diagonal matrix:
[[1 0 0 0]
 [0 2 0 0]
 [0 0 0 0]
 [0 0 0 3]]
Diagonal elements: [1 2 0 3]
Rank: 3
Expected: 3 (number of non-zero diagonal elements)

4. Rank-1 Matrix
u = 
[[1]
 [2]
 [3]]
v = 
[[2]
 [3]
 [4]]
A = u @ v^T = 
[[ 2  3  4]
 [ 4  6  8]
 [ 6  9 12]]
Rank of A: 1
Expected: 1
Row ratios:
Row 1 / Row 0 = [2. 2. 2.]
Row 2 / Row 0 = [3. 3. 3.]

5. Rank of Transpose
Matrix A (2x3):
[[1 2 3]
 [4 5 6]]
rank(A) = 2
rank(A^T) = 2
They are equal: True

6. Rank of Matrix Products
Matrix A (3x2), rank = 2
Matrix B (2x3), rank = 2
A @ B (3x3), rank = 2
rank(AB) ≤ min(rank(A), rank(B)) = min(2, 2) = 2
Verified: True


### **3.2 Rank of Added and Multiplied Matrices**

When matrices are added or multiplied, their ranks follow specific relationships:

**Matrix Addition**: rank(A + B) ≤ rank(A) + rank(B). The rank of a sum is at most the sum of the ranks. However, if the matrices have overlapping column spaces, the rank of the sum could be significantly less than the sum of individual ranks.

**Matrix Multiplication**: rank(AB) = rank(A) if rank(B) = number of columns of A (B has full row rank). Similarly, rank(AB) = rank(B) if rank(A) = number of rows of A (A has full column rank).

**Multiplication by Full Rank Matrices**: If C is a square full-rank matrix, then rank(CA) = rank(A) and rank(AC) = rank(A). Multiplying by a full-rank matrix preserves the rank.

These properties are fundamental for understanding how operations affect the dimensionality and structure of matrices in data science applications.

In [12]:
print("Rank of Added and Multiplied Matrices\n")
print("=" * 50)

# Example: Rank of sums
print("\n1. Rank of Matrix Addition")
A = np.array([[1, 0],
              [0, 0]])
B = np.array([[0, 0],
              [0, 1]])

print(f"Matrix A:\n{A}")
print(f"rank(A) = {np.linalg.matrix_rank(A)}")
print(f"\nMatrix B:\n{B}")
print(f"rank(B) = {np.linalg.matrix_rank(B)}")
print(f"\nA + B:\n{A + B}")
print(f"rank(A + B) = {np.linalg.matrix_rank(A + B)}")
print(f"rank(A) + rank(B) = {np.linalg.matrix_rank(A) + np.linalg.matrix_rank(B)}")
print(f"rank(A + B) ≤ rank(A) + rank(B): {np.linalg.matrix_rank(A + B) <= np.linalg.matrix_rank(A) + np.linalg.matrix_rank(B)}")

# Example with overlapping column spaces
print("\n2. Addition with Overlapping Column Spaces")
A = np.array([[1, 2],
              [2, 4]])
B = np.array([[2, 4],
              [4, 8]])

print(f"Matrix A:\n{A}")
print(f"rank(A) = {np.linalg.matrix_rank(A)}")
print(f"\nMatrix B (B = 2*A):\n{B}")
print(f"rank(B) = {np.linalg.matrix_rank(B)}")
print(f"\nA + B:\n{A + B}")
print(f"rank(A + B) = {np.linalg.matrix_rank(A + B)}")
print(f"rank(A) + rank(B) = {np.linalg.matrix_rank(A) + np.linalg.matrix_rank(B)}")
print(f"When matrices have same column space, rank(A+B) < rank(A) + rank(B)")

# Example: Rank of products
print("\n3. Rank of Matrix Products")
A = np.array([[1, 2],
              [3, 4],
              [5, 6]])
print(f"Matrix A (3x2):\n{A}")
print(f"rank(A) = {np.linalg.matrix_rank(A)}")

B = np.array([[1, 0, 1],
              [0, 1, 1]])
print(f"\nMatrix B (2x3):\n{B}")
print(f"rank(B) = {np.linalg.matrix_rank(B)}")
print(f"B has full row rank: {np.linalg.matrix_rank(B) == B.shape[0]}")

AB = A @ B
print(f"\nA @ B (3x3):\n{AB}")
print(f"rank(A @ B) = {np.linalg.matrix_rank(AB)}")
print(f"Since B has full row rank (rank=2=# cols of A), rank(AB) = rank(A) = {np.linalg.matrix_rank(A)}")
print(f"Verified: {np.linalg.matrix_rank(AB) == np.linalg.matrix_rank(A)}")

# Example: Multiplication by full rank matrix
print("\n4. Multiplication by Full Rank Matrix")
C = np.array([[2, 1],
              [1, 2]])  # Full rank 2x2 matrix
A_test = np.array([[1, 2],
                  [3, 4],
                  [5, 6]])

print(f"Full rank matrix C (2x2):\n{C}")
print(f"rank(C) = {np.linalg.matrix_rank(C)}")
print(f"\nMatrix A (3x2):\n{A_test}")
print(f"rank(A) = {np.linalg.matrix_rank(A_test)}")

CA = C @ A_test.T  # Note: need to transpose for compatibility
print(f"\nC @ A^T (2x3):\n{CA}")
print(f"rank(C @ A^T) = {np.linalg.matrix_rank(CA)}")
print(f"Multiplying by full rank matrix preserves rank structure")

Rank of Added and Multiplied Matrices


1. Rank of Matrix Addition
Matrix A:
[[1 0]
 [0 0]]
rank(A) = 1

Matrix B:
[[0 0]
 [0 1]]
rank(B) = 1

A + B:
[[1 0]
 [0 1]]
rank(A + B) = 2
rank(A) + rank(B) = 2
rank(A + B) ≤ rank(A) + rank(B): True

2. Addition with Overlapping Column Spaces
Matrix A:
[[1 2]
 [2 4]]
rank(A) = 1

Matrix B (B = 2*A):
[[2 4]
 [4 8]]
rank(B) = 1

A + B:
[[ 3  6]
 [ 6 12]]
rank(A + B) = 1
rank(A) + rank(B) = 2
When matrices have same column space, rank(A+B) < rank(A) + rank(B)

3. Rank of Matrix Products
Matrix A (3x2):
[[1 2]
 [3 4]
 [5 6]]
rank(A) = 2

Matrix B (2x3):
[[1 0 1]
 [0 1 1]]
rank(B) = 2
B has full row rank: True

A @ B (3x3):
[[ 1  2  3]
 [ 3  4  7]
 [ 5  6 11]]
rank(A @ B) = 2
Since B has full row rank (rank=2=# cols of A), rank(AB) = rank(A) = 2
Verified: True

4. Multiplication by Full Rank Matrix
Full rank matrix C (2x2):
[[2 1]
 [1 2]]
rank(C) = 2

Matrix A (3x2):
[[1 2]
 [3 4]
 [5 6]]
rank(A) = 2

C @ A^T (2x3):
[[ 4 10 16]
 [ 5 11 17]]
rank(C @

### **3.3 Rank of Shifted Matrices**

A shifted matrix is obtained by adding a scalar multiple of the identity matrix to a given matrix: A' = A + λI. This operation increases the diagonal elements by λ while leaving off-diagonal elements unchanged.

The rank of a shifted matrix depends on the magnitude of the shift and the properties of the original matrix. In general:

**Shifted Singular Matrix**: If A is singular (rank < n) with rank deficiency r, then for most values of λ, the shifted matrix A + λI will be full rank. However, for certain special values of λ (specifically, λ = -λᵢ where λᵢ is an eigenvalue of A), the shifted matrix may remain singular.

**Application in Eigenvalues**: Matrix shifting is intimately related to computing eigenvalues. The eigenvalues λ of matrix A are precisely those values for which det(A - λI) = 0, meaning that A - λI is singular.

**Application in Regularization**: In machine learning and statistics, small shifts are used to regularize matrices, making them numerically stable when they are near-singular.

In [13]:
print("Rank of Shifted Matrices\n")
print("=" * 50)

# Example 1: Shifting a singular matrix
print("\nExample 1: Shifting a singular matrix")
A_singular = np.array([[1, 2],
                      [2, 4]])  # Rank-1 matrix
print(f"Matrix A (singular):\n{A_singular}")
print(f"rank(A) = {np.linalg.matrix_rank(A_singular)}")
print(f"det(A) = {np.linalg.det(A_singular):.10f}")

print(f"\nShifted matrices A + λI:")
for lam in [0.1, 1, 2, 10]:
    A_shifted = A_singular + lam * np.eye(A_singular.shape[0])
    rank_shifted = np.linalg.matrix_rank(A_shifted)
    det_shifted = np.linalg.det(A_shifted)
    print(f"\nλ = {lam}:")
    print(f"  rank(A + {lam}I) = {rank_shifted}")
    print(f"  det(A + {lam}I) = {det_shifted:.6f}")
    print(f"  Is full rank: {rank_shifted == 2}")

# Example 2: Shifting and eigenvalues
print("\n" + "=" * 50)
print("\nExample 2: Connection to Eigenvalues")
A = np.array([[3, 1],
              [1, 3]])
print(f"Matrix A:\n{A}")

# Get eigenvalues
eigenvalues = np.linalg.eigvals(A)
print(f"\nEigenvalues of A: {eigenvalues}")

print(f"\nFor shifted matrix (A - λI), rank becomes deficient when λ is an eigenvalue:")
for lam in [1, 2, 4, eigenvalues[0], eigenvalues[1]]:
    A_shifted = A - lam * np.eye(A.shape[0])
    rank_shifted = np.linalg.matrix_rank(A_shifted, tol=1e-10)
    det_shifted = np.linalg.det(A_shifted)
    print(f"\nλ = {lam:.4f}:")
    print(f"  rank(A - {lam:.4f}I) = {rank_shifted}")
    print(f"  det(A - {lam:.4f}I) = {det_shifted:.10f}")
    if rank_shifted < A.shape[0]:
        print(f"  >>> Singular! λ = {lam:.4f} is an eigenvalue")

# Example 3: Regularization effect
print("\n" + "=" * 50)
print("\nExample 3: Regularization in Practice")
print("\nNumerical stability: condition number of matrices")
A_ill_conditioned = np.array([[1e-10, 0],
                             [0, 1]])  # Very poorly conditioned
print(f"\nOriginal matrix condition number: {np.linalg.cond(A_ill_conditioned):.2e}")

for lam in [1e-10, 1e-5, 0.01, 0.1]:
    A_shifted = A_ill_conditioned + lam * np.eye(A_ill_conditioned.shape[0])
    cond = np.linalg.cond(A_shifted)
    print(f"Condition number with λ = {lam}: {cond:.2e}")

print(f"\nSmall shifts improve numerical stability!")

Rank of Shifted Matrices


Example 1: Shifting a singular matrix
Matrix A (singular):
[[1 2]
 [2 4]]
rank(A) = 1
det(A) = 0.0000000000

Shifted matrices A + λI:

λ = 0.1:
  rank(A + 0.1I) = 2
  det(A + 0.1I) = 0.510000
  Is full rank: True

λ = 1:
  rank(A + 1I) = 2
  det(A + 1I) = 6.000000
  Is full rank: True

λ = 2:
  rank(A + 2I) = 2
  det(A + 2I) = 14.000000
  Is full rank: True

λ = 10:
  rank(A + 10I) = 2
  det(A + 10I) = 150.000000
  Is full rank: True


Example 2: Connection to Eigenvalues
Matrix A:
[[3 1]
 [1 3]]

Eigenvalues of A: [4. 2.]

For shifted matrix (A - λI), rank becomes deficient when λ is an eigenvalue:

λ = 1.0000:
  rank(A - 1.0000I) = 2
  det(A - 1.0000I) = 3.0000000000

λ = 2.0000:
  rank(A - 2.0000I) = 1
  det(A - 2.0000I) = 0.0000000000
  >>> Singular! λ = 2.0000 is an eigenvalue

λ = 4.0000:
  rank(A - 4.0000I) = 1
  det(A - 4.0000I) = 0.0000000000
  >>> Singular! λ = 4.0000 is an eigenvalue

λ = 4.0000:
  rank(A - 4.0000I) = 1
  det(A - 4.0000I) = 0.00000

## **4. Determinant**

The determinant of a square matrix is a single number that encodes important information about the matrix. The determinant of a matrix A is denoted det(A) or |A|. The determinant has several important interpretations:

**Invertibility**: A square matrix is invertible if and only if its determinant is non-zero (det(A) ≠ 0). If det(A) = 0, the matrix is singular and does not have an inverse.

**Volume Scaling**: The absolute value of the determinant |det(A)| represents the factor by which a linear transformation (represented by matrix A) scales volumes in n-dimensional space.

**Orientation**: The sign of the determinant indicates whether the transformation preserves (positive) or reverses (negative) orientation.

**Linear Independence**: A matrix has full rank (columns are linearly independent) if and only if its determinant is non-zero.

Computing determinants for large matrices is computationally expensive, but for small matrices and for theoretical work, the determinant provides valuable insight into matrix properties.

In [14]:
print("Computing Determinants\n")
print("=" * 50)

# 2x2 matrix
print("\n2x2 Matrix:")
A2x2 = np.array([[3, 8],
                [4, 6]])
print(f"A = \n{A2x2}")
# For 2x2: det(A) = ad - bc
det_2x2_manual = 3*6 - 8*4
det_2x2_numpy = np.linalg.det(A2x2)
print(f"\nManual calculation: det(A) = 3*6 - 8*4 = {det_2x2_manual}")
print(f"NumPy: det(A) = {det_2x2_numpy}")

# 3x3 matrix
print("\n\n3x3 Matrix:")
A3x3 = np.array([[1, 2, 3],
                [0, 4, 5],
                [1, 0, 6]])
print(f"A = \n{A3x3}")
det_3x3 = np.linalg.det(A3x3)
print(f"\ndet(A) = {det_3x3:.6f}")

# 4x4 matrix
print("\n\n4x4 Matrix:")
A4x4 = np.random.randn(4, 4)
print(f"A = \n{A4x4}")
det_4x4 = np.linalg.det(A4x4)
print(f"\ndet(A) = {det_4x4:.6f}")

# Determinants of special matrices
print("\n" + "=" * 50)
print("\nDeterminants of Special Matrices:\n")

# Identity matrix
I = np.eye(3)
print(f"det(I) = {np.linalg.det(I):.4f} (always 1)")

# Zero matrix
Z = np.zeros((3, 3))
print(f"det(0) = {np.linalg.det(Z):.4f} (always 0)")

# Diagonal matrix
D = np.diag([2, 3, 4])
print(f"det(diag) = {np.linalg.det(D):.4f} = 2*3*4 = {2*3*4}")

# Triangular matrix
L = np.array([[2, 0, 0],
              [3, 4, 0],
              [5, 6, 7]])
print(f"det(triangular) = {np.linalg.det(L):.4f} = 2*4*7 = {2*4*7} (product of diagonal)")

# Singular matrix (det = 0)
S = np.array([[1, 2, 3],
              [2, 4, 6],
              [3, 6, 9]])
print(f"det(singular) = {np.linalg.det(S):.10f} ≈ 0 (linearly dependent rows)")

Computing Determinants


2x2 Matrix:
A = 
[[3 8]
 [4 6]]

Manual calculation: det(A) = 3*6 - 8*4 = -14
NumPy: det(A) = -14.000000000000004


3x3 Matrix:
A = 
[[1 2 3]
 [0 4 5]
 [1 0 6]]

det(A) = 22.000000


4x4 Matrix:
A = 
[[ 0.49671415 -0.1382643   0.64768854  1.52302986]
 [-0.23415337 -0.23413696  1.57921282  0.76743473]
 [-0.46947439  0.54256004 -0.46341769 -0.46572975]
 [ 0.24196227 -1.91328024 -1.72491783 -0.56228753]]

det(A) = -1.863820


Determinants of Special Matrices:

det(I) = 1.0000 (always 1)
det(0) = 0.0000 (always 0)
det(diag) = 24.0000 = 2*3*4 = 24
det(triangular) = 56.0000 = 2*4*7 = 56 (product of diagonal)
det(singular) = 0.0000000000 ≈ 0 (linearly dependent rows)


### **4.1 Determinant with Linear Dependencies**

One of the most important properties of the determinant is its relationship to linear independence. If a matrix has linearly dependent rows or columns, its determinant is exactly zero. Conversely, if det(A) ≠ 0, then all rows and columns are linearly independent.

This relationship makes the determinant a powerful tool for testing linear independence. When computing determinants numerically, very small values close to zero (like 1e-15) should be treated as zero due to computer precision errors.

For matrices with linearly dependent rows or columns:
- The rank is less than n (where n is the size of the square matrix)
- The determinant is zero
- The matrix is singular and not invertible
- The null space contains non-zero vectors

In [15]:
print("Determinant and Linear Dependence\n")
print("=" * 50)

# Example 1: Linearly dependent rows
print("\nExample 1: Matrix with linearly dependent rows")
A = np.array([[1, 2, 3],
              [2, 4, 6],  # 2 * row 1
              [3, 6, 9]])  # 3 * row 1

print(f"Matrix A:\n{A}")
print(f"\nRow 2 = 2 * Row 1: {np.allclose(A[1], 2*A[0])}")
print(f"Row 3 = 3 * Row 1: {np.allclose(A[2], 3*A[0])}")

det_A = np.linalg.det(A)
rank_A = np.linalg.matrix_rank(A)
print(f"\ndet(A) = {det_A:.15f}")
print(f"rank(A) = {rank_A}")
print(f"det(A) ≈ 0: {np.isclose(det_A, 0)}")
print(f"Matrix is singular: {rank_A < A.shape[0]}")

# Example 2: Linearly independent rows
print("\n" + "=" * 50)
print("\nExample 2: Matrix with linearly independent rows")
B = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 10]])

print(f"Matrix B:\n{B}")
det_B = np.linalg.det(B)
rank_B = np.linalg.matrix_rank(B)
print(f"\ndet(B) = {det_B:.6f}")
print(f"rank(B) = {rank_B}")
print(f"det(B) ≠ 0: {not np.isclose(det_B, 0)}")
print(f"Matrix is invertible: {rank_B == B.shape[0]}")

# Example 3: Almost singular matrix (numerical precision)
print("\n" + "=" * 50)
print("\nExample 3: Near-singular matrix (numerical challenge)")
C = np.array([[1, 2],
              [1, 2.0000000001]])  # Almost identical rows

print(f"Matrix C:\n{C}")
det_C = np.linalg.det(C)
print(f"\ndet(C) = {det_C:.15e}")
print(f"This is technically non-zero, but very close to zero")
print(f"\nDetecting this numerically is challenging!")
print(f"We should use rank instead of determinant for numerical stability")
rank_C = np.linalg.matrix_rank(C, tol=1e-8)
print(f"rank(C) with tolerance 1e-8: {rank_C}")

Determinant and Linear Dependence


Example 1: Matrix with linearly dependent rows
Matrix A:
[[1 2 3]
 [2 4 6]
 [3 6 9]]

Row 2 = 2 * Row 1: True
Row 3 = 3 * Row 1: True

det(A) = 0.000000000000000
rank(A) = 1
det(A) ≈ 0: True
Matrix is singular: True


Example 2: Matrix with linearly independent rows
Matrix B:
[[ 1  2  3]
 [ 4  5  6]
 [ 7  8 10]]

det(B) = -3.000000
rank(B) = 3
det(B) ≠ 0: True
Matrix is invertible: True


Example 3: Near-singular matrix (numerical challenge)
Matrix C:
[[1. 2.]
 [1. 2.]]

det(C) = 1.000000082740370e-10
This is technically non-zero, but very close to zero

Detecting this numerically is challenging!
We should use rank instead of determinant for numerical stability
rank(C) with tolerance 1e-8: 1


### **4.2 The Characteristic Polynomial**

The characteristic polynomial of a matrix A is defined as:

det(A - λI) = 0

where λ is a scalar and I is the identity matrix. The solutions to this polynomial equation are the eigenvalues of the matrix. The characteristic polynomial is fundamental to finding eigenvalues, which are used extensively in data science for dimension reduction (principal component analysis), stability analysis, and many other applications.

For an n×n matrix, the characteristic polynomial has degree n, meaning there are exactly n eigenvalues (counting multiplicities and including complex eigenvalues). The characteristic polynomial can be written as:

det(A - λI) = (-1)^n λ^n + ... + det(A)

Notice that:
- The constant term is det(A)
- The sum of eigenvalues equals the trace of A
- The product of eigenvalues equals the determinant of A

Computing the characteristic polynomial is the first step in finding eigenvalues, though modern computer algorithms typically use more efficient methods like the QR algorithm.

In [16]:
print("Characteristic Polynomial\n")
print("=" * 50)

# Example 1: 2x2 matrix
print("\nExample 1: 2x2 Matrix")
A = np.array([[3, 1],
              [1, 3]])
print(f"Matrix A:\n{A}")
print(f"\nCharacteristic polynomial: det(A - λI) = 0")
print(f"det([[3-λ, 1], [1, 3-λ]]) = 0")
print(f"(3-λ)(3-λ) - 1*1 = 0")
print(f"9 - 6λ + λ² - 1 = 0")
print(f"λ² - 6λ + 8 = 0")
print(f"(λ - 2)(λ - 4) = 0")

# Compute eigenvalues using numpy
eigenvalues = np.linalg.eigvals(A)
print(f"\nEigenvalues: {eigenvalues}")

# Verify
for lam in eigenvalues:
    char_poly = np.linalg.det(A - lam * np.eye(A.shape[0]))
    print(f"det(A - {lam:.4f}I) = {char_poly:.10f}")

print(f"\nProperties:")
print(f"Sum of eigenvalues = {np.sum(eigenvalues):.4f}")
print(f"Trace of A = {np.trace(A):.4f}")
print(f"Product of eigenvalues = {np.prod(eigenvalues):.4f}")
print(f"det(A) = {np.linalg.det(A):.4f}")

# Example 2: 3x3 matrix
print("\n" + "=" * 50)
print("\nExample 2: 3x3 Matrix")
B = np.array([[2, 0, 0],
              [0, 3, 0],
              [0, 0, 5]])
print(f"Matrix B (diagonal):\n{B}")
print(f"\nFor diagonal matrices, eigenvalues are the diagonal elements")

eigenvalues_B = np.linalg.eigvals(B)
print(f"Eigenvalues: {eigenvalues_B}")
print(f"Diagonal elements: {np.diag(B)}")

print(f"\nVerifying characteristic polynomial properties:")
print(f"Sum of eigenvalues = {np.sum(eigenvalues_B):.4f}")
print(f"Trace of B = {np.trace(B):.4f}")
print(f"Product of eigenvalues = {np.prod(eigenvalues_B):.4f}")
print(f"det(B) = {np.linalg.det(B):.4f}")

# Example 3: Using numpy.poly to compute characteristic polynomial
print("\n" + "=" * 50)
print("\nExample 3: Computing Characteristic Polynomial Coefficients")
C = np.array([[1, 2],
              [3, 4]])
print(f"Matrix C:\n{C}")

# poly returns coefficients of polynomial, highest degree first
char_poly_coeffs = np.poly(C)  # Returns coefficients of det(C - λI)
print(f"\nCharacteristic polynomial coefficients (highest degree first): {char_poly_coeffs}")
print(f"This means: λ² - 5λ - 2")

# Verify by computing eigenvalues from roots
eigenvalues_C = np.roots(char_poly_coeffs)
print(f"\nEigenvalues (from polynomial roots): {eigenvalues_C}")
print(f"Eigenvalues (from numpy.linalg.eigvals): {np.linalg.eigvals(C)}")

Characteristic Polynomial


Example 1: 2x2 Matrix
Matrix A:
[[3 1]
 [1 3]]

Characteristic polynomial: det(A - λI) = 0
det([[3-λ, 1], [1, 3-λ]]) = 0
(3-λ)(3-λ) - 1*1 = 0
9 - 6λ + λ² - 1 = 0
λ² - 6λ + 8 = 0
(λ - 2)(λ - 4) = 0

Eigenvalues: [4. 2.]
det(A - 4.0000I) = 0.0000000000
det(A - 2.0000I) = 0.0000000000

Properties:
Sum of eigenvalues = 6.0000
Trace of A = 6.0000
Product of eigenvalues = 8.0000
det(A) = 8.0000


Example 2: 3x3 Matrix
Matrix B (diagonal):
[[2 0 0]
 [0 3 0]
 [0 0 5]]

For diagonal matrices, eigenvalues are the diagonal elements
Eigenvalues: [2. 3. 5.]
Diagonal elements: [2 3 5]

Verifying characteristic polynomial properties:
Sum of eigenvalues = 10.0000
Trace of B = 10.0000
Product of eigenvalues = 30.0000
det(B) = 30.0000


Example 3: Computing Characteristic Polynomial Coefficients
Matrix C:
[[1 2]
 [3 4]]

Characteristic polynomial coefficients (highest degree first): [ 1. -5. -2.]
This means: λ² - 5λ - 2

Eigenvalues (from polynomial roots): [ 5.37228132 -0.37

## **5. Summary**

This chapter has introduced you to the more advanced aspects of matrix operations that form the foundation for understanding linear algebra beyond basic calculations. Here are the key takeaways:

**Matrix Norms**: Matrix norms provide a single number that characterizes the magnitude of a matrix. The Frobenius norm (ℓ2 norm) is the most commonly used element-wise norm in data science and is fundamental to regularization techniques in machine learning.

**Matrix Spaces**: The four fundamental matrix spaces—column space, row space, null space, and left null space—describe the structure of a matrix. These spaces are fundamental to understanding which systems of linear equations have solutions and what those solutions look like.

**Rank**: The rank of a matrix is the dimension of its column (and row) space. Rank determines invertibility, the solvability of linear systems, and is central to nearly every advanced application of linear algebra in data science.

**Determinant**: The determinant is a single number that encodes information about linear independence, invertibility, and volume scaling. For square full-rank matrices (det(A) ≠ 0), the matrix is invertible.

**Characteristic Polynomial**: The characteristic polynomial det(A - λI) = 0 provides a method for finding eigenvalues, which are crucial for understanding and analyzing the behavior of matrices in applications.

These concepts are the bridges between elementary linear algebra and the advanced topics of inverse, decompositions, and data science applications that will follow in subsequent chapters. Mastering these concepts is essential for becoming proficient in applied linear algebra.